# GTE embeddings benchmark — French vs. English (STS)

Benchmarks the Databricks Foundation Model API endpoint **`databricks-gte-large-en`**
on the **STS Benchmark**, run in parallel on **English** and **French**, to measure how
much embedding quality the English-tuned GTE model loses on French.

- **Dataset:** `stsb_multi_mt` (`en` and `fr` configs) — the same 1,379 STS-B test pairs
  with identical gold similarity scores (0–5). This is MTEB's French STS task.
- **Metric:** Spearman correlation between cosine similarity and gold score
  (`cosine_spearman`, MTEB's standard STS metric), reported per language + FR/EN ratio.
- **Compute:** serverless. **Model access:** FMAPI only (no local model).
- **Data caching:** downloaded once from HuggingFace into a UC Volume; subsequent runs
  read the cached parquet and skip the download.

See `SPEC/SPECS.md` for the full specification.

In [ ]:
# `mlflow.deployments` is the FMAPI client and is not preinstalled on serverless.
# Use mlflow-skinny (lightweight, no numpy/pandas pins). Do NOT pass -U: upgrading
# numpy breaks serverless's prebuilt pandas/pyspark.
%pip install -q mlflow-skinny
dbutils.library.restartPython()

In [ ]:
# --- Config ---
ENDPOINT = "databricks-gte-large-en"       # Databricks FMAPI embedding endpoint
HF_DATASET = "PhilipMay/stsb_multi_mt"     # parallel multilingual STS-B (namespaced repo id)
LANGS = ["en", "fr"]
SPLIT = "test"                             # 1,379 pairs per language
GOLD_MAX = 5.0                             # STS-B score range is 0..5
BATCH_SIZE = 100                           # texts per FMAPI request

# UC Volume used as the dataset cache. Edit catalog/schema to taste.
CATALOG = "lucasbruand_catalog"
SCHEMA = "gte_french_bench"
VOLUME = "data"
VOLUME_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/stsb"
RESULTS_CSV = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/results_gte_french.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
import os
os.makedirs(VOLUME_DIR, exist_ok=True)
print(f"Cache dir: {VOLUME_DIR}")

In [ ]:
# --- Load STS-B per language, caching to the UC Volume ---
# Download the parquet directly from HuggingFace's parquet API (no `datasets` dependency,
# avoids the serverless /root/.cache permission issue). Download only on cache miss.
import os
import io
import urllib.request
import pandas as pd

def load_sts(lang):
    cache_path = os.path.join(VOLUME_DIR, f"stsb_{lang}_{SPLIT}.parquet")
    if os.path.exists(cache_path):
        print(f"[{lang}] cache hit  -> {cache_path}")
        return pd.read_parquet(cache_path)
    url = f"https://huggingface.co/api/datasets/{HF_DATASET}/parquet/{lang}/{SPLIT}/0.parquet"
    print(f"[{lang}] cache miss -> downloading {url}")
    raw = urllib.request.urlopen(url, timeout=60).read()
    df = pd.read_parquet(io.BytesIO(raw))[["sentence1", "sentence2", "similarity_score"]]
    df.to_parquet(cache_path, index=False)
    print(f"[{lang}] cached {len(df)} rows -> {cache_path}")
    return df

data = {lang: load_sts(lang) for lang in LANGS}
for lang, df in data.items():
    print(f"[{lang}] {df.shape[0]} pairs, score range {df.similarity_score.min()}..{df.similarity_score.max()}")

In [ ]:
# --- FMAPI embedding helper (MLflow deployments client) ---
import time
import numpy as np
from mlflow.deployments import get_deploy_client

_client = get_deploy_client("databricks")

def embed(texts, endpoint=ENDPOINT, batch_size=BATCH_SIZE, max_retries=5):
    """Return an (n, dim) float32 array of L2-normalized embeddings."""
    out = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(max_retries):
            try:
                resp = _client.predict(endpoint=endpoint, inputs={"input": batch})
                out.extend(item["embedding"] for item in resp["data"])
                break
            except Exception as e:  # rate limit / transient — backoff and retry
                if attempt == max_retries - 1:
                    raise
                wait = 2 ** attempt
                print(f"  batch {start} attempt {attempt+1} failed ({e}); retry in {wait}s")
                time.sleep(wait)
    arr = np.asarray(out, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return arr / norms

# Smoke test
_probe = embed(["hello world", "bonjour le monde"])
print(f"Embedding dim: {_probe.shape[1]}")

In [ ]:
# --- Run the STS benchmark for each language ---
from scipy.stats import spearmanr, pearsonr

rows = []
for lang in LANGS:
    df = data[lang]
    s1 = df["sentence1"].tolist()
    s2 = df["sentence2"].tolist()
    gold = df["similarity_score"].to_numpy(dtype=np.float32) / GOLD_MAX
    print(f"[{lang}] {len(s1)} pairs — embedding...")

    e1 = embed(s1)
    e2 = embed(s2)
    cos = np.sum(e1 * e2, axis=1)  # both sides already L2-normalized

    spearman = spearmanr(cos, gold).correlation
    pearson = pearsonr(cos, gold)[0]
    rows.append({"lang": lang, "n_pairs": len(s1),
                 "cosine_spearman": spearman, "cosine_pearson": pearson})
    print(f"[{lang}] cosine_spearman={spearman:.4f}  cosine_pearson={pearson:.4f}")

results = pd.DataFrame(rows).set_index("lang")

In [ ]:
# --- Headline: FR/EN ratio + results table ---
ratio = results.loc["fr", "cosine_spearman"] / results.loc["en", "cosine_spearman"]

print("=" * 56)
print(f"Endpoint: {ENDPOINT}")
print(f"Dataset:  {HF_DATASET} [{SPLIT}]  (cached in {VOLUME_DIR})")
print("=" * 56)
print(results.round(4).to_string())
print("-" * 56)
print(f"FR/EN cosine_spearman ratio: {ratio:.3f}")
print("  ~1.0  -> French served about as well as English")
print("  <<1.0 -> English GTE endpoint is a poor fit for French")

results.to_csv(RESULTS_CSV)
print(f"\nSaved: {RESULTS_CSV}")
display(results.reset_index())